# Notebook 03 — Baselines intra-dataset

**Projet :** Generalization of Intrusion Detection Systems across datasets  
**Datasets :** CIC-IDS2017 et UNSW-NB15  
**Objectif :** établir des baselines reproductibles sur chaque dataset avant l'évaluation cross-dataset.

---

## 1. Objectifs et protocole

Ce notebook entraîne et évalue trois modèles supervisés :

- Régression logistique ;
- Random Forest ;
- MLP (réseau de neurones entièrement connecté).

Pour chaque dataset, le protocole utilise les cinq partitions externes sauvegardées dans `results/tables/*_group_splits_v2.npz`. Chaque partition externe est évaluée avec trois graines aléatoires (`42`, `123`, `2026`).

**Évaluation :**
- CIC-IDS2017 entraînement/test sur CIC-IDS2017 ;
- UNSW-NB15 entraînement/test sur UNSW-NB15.

Le seuil de classification est sélectionné sur une validation interne issue exclusivement de la partie entraînement du fold externe. Le fold externe de test n'intervient ni dans la sélection du seuil ni dans l'ajustement du modèle.

**Prévention des fuites :**
- séparation groupée à partir des groupes d'observations ayant exactement les mêmes cinq caractéristiques ;
- vérification de l'absence de groupes communs entre entraînement, validation et test ;
- standardisation ajustée uniquement sur l'entraînement, via `Pipeline` ;
- aucun rééchantillonnage avant séparation ;
- seuil optimisé sur la validation, jamais sur le test.

> **Limite à garder en tête :** les groupes représentent les doublons exacts sur les cinq variables retenues. Ils ne représentent pas des sessions, hôtes ou flux réseau indépendants. Ce protocole réduit donc le risque de fuite par doublons exacts, sans garantir l'indépendance au niveau réseau.

Les résultats constituent des baselines sur un échantillon harmonisé de 100 000 observations par dataset. Ils ne doivent pas être présentés comme des résultats obtenus sur l'intégralité des jeux de données.


## 2. Configuration

Exécuter les cellules dans l'ordre. Le notebook détecte automatiquement la racine du projet si le répertoire courant est le projet ou `notebooks/`.


In [1]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
)
from sklearn.exceptions import ConvergenceWarning

# Les avertissements de convergence du MLP restent visibles : ils doivent être examinés.
warnings.filterwarnings("default", category=ConvergenceWarning)

SEEDS = [42, 123, 2026]
N_OUTER_FOLDS = 5
N_INNER_FOLDS = 5

# True : exécuter toute l'expérience (2 datasets x 3 modèles x 5 folds x 3 seeds).
# False : exécuter seulement un essai rapide défini plus bas.
RUN_FULL_EXPERIMENT = True

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT_DIR / "data" / "interim"
TABLES_DIR = ROOT_DIR / "results" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = [
    "flow_duration_s",
    "fwd_packets",
    "bwd_packets",
    "fwd_bytes",
    "bwd_bytes",
]
TARGET_COLUMN = "target"

print("Racine du projet :", ROOT_DIR)
print("Données :", DATA_DIR)
print("Résultats :", TABLES_DIR)
print("Seeds :", SEEDS)
print("Nombre de folds externes :", N_OUTER_FOLDS)
print("Exécution complète :", RUN_FULL_EXPERIMENT)


Racine du projet : D:\Master-SDIA\CYB-1-IDS-Generalization
Données : D:\Master-SDIA\CYB-1-IDS-Generalization\data\interim
Résultats : D:\Master-SDIA\CYB-1-IDS-Generalization\results\tables
Seeds : [42, 123, 2026]
Nombre de folds externes : 5
Exécution complète : True


## 3. Chargement des échantillons harmonisés et des folds

Les fichiers chargés sont ceux produits par le notebook `02_dataset_harmonization.ipynb` :

- `cic_harmonized_sample_v2.csv`
- `unsw_harmonized_sample_v2.csv`
- `cic_group_splits_v2.npz`
- `unsw_group_splits_v2.npz`

Les identifiants de groupes sont reconstruits avec la même règle : factorisation des combinaisons exactes des cinq caractéristiques.


In [2]:
def load_dataset(dataset_name, sample_filename, splits_filename):
    sample_path = DATA_DIR / sample_filename
    splits_path = TABLES_DIR / splits_filename

    if not sample_path.exists():
        raise FileNotFoundError(f"Échantillon introuvable : {sample_path}")
    if not splits_path.exists():
        raise FileNotFoundError(f"Fichier de folds introuvable : {splits_path}")

    df = pd.read_csv(sample_path)

    required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f"{dataset_name} : colonnes manquantes : {missing}")

    X = df[FEATURE_COLUMNS].copy()
    y = df[TARGET_COLUMN].astype(int).to_numpy()

    # Même définition des groupes que dans le notebook 02.
    feature_index = pd.MultiIndex.from_frame(X)
    groups, _ = pd.factorize(feature_index, sort=False)
    groups = np.asarray(groups)

    with np.load(splits_path) as split_data:
        outer_splits = [
            (
                split_data[f"train_{fold}"].astype(int).copy(),
                split_data[f"test_{fold}"].astype(int).copy(),
            )
            for fold in range(1, N_OUTER_FOLDS + 1)
        ]

    if len(X) != len(y) or len(y) != len(groups):
        raise AssertionError("Dimensions incohérentes entre X, y et groups.")
    if not set(np.unique(y)).issubset({0, 1}):
        raise ValueError("La cible doit être binaire et codée 0/1.")
    if not np.isfinite(X.to_numpy(dtype=float)).all():
        raise ValueError(f"{dataset_name} contient des valeurs non finies.")
    if len(outer_splits) != N_OUTER_FOLDS:
        raise AssertionError("Nombre de folds externes inattendu.")

    for fold_number, (train_idx, test_idx) in enumerate(outer_splits, start=1):
        if len(np.intersect1d(train_idx, test_idx)) != 0:
            raise AssertionError(f"Chevauchement d'indices dans le fold {fold_number}.")
        if len(np.intersect1d(np.unique(groups[train_idx]), np.unique(groups[test_idx]))) != 0:
            raise AssertionError(f"Fuite de groupes dans le fold {fold_number}.")

    print(f"\n{dataset_name}")
    print("  X :", X.shape)
    print("  Groupes uniques :", len(np.unique(groups)))
    print("  Classes :", dict(zip(*np.unique(y, return_counts=True))))
    print("  Folds externes vérifiés :", len(outer_splits))

    return {
        "name": dataset_name,
        "X": X,
        "y": y,
        "groups": groups,
        "outer_splits": outer_splits,
    }


cic_data = load_dataset(
    "CIC-IDS2017",
    "cic_harmonized_sample_v2.csv",
    "cic_group_splits_v2.npz",
)

unsw_data = load_dataset(
    "UNSW-NB15",
    "unsw_harmonized_sample_v2.csv",
    "unsw_group_splits_v2.npz",
)



CIC-IDS2017
  X : (100000, 5)
  Groupes uniques : 74294
  Classes : {np.int64(0): np.int64(80212), np.int64(1): np.int64(19788)}
  Folds externes vérifiés : 5

UNSW-NB15
  X : (100000, 5)
  Groupes uniques : 64117
  Classes : {np.int64(0): np.int64(87423), np.int64(1): np.int64(12577)}
  Folds externes vérifiés : 5


## 4. Définition des modèles

Les hyperparamètres sont fixés à l'avance pour établir une première comparaison de baselines, et non pour prétendre à une optimisation exhaustive.

- La régression logistique et le MLP sont standardisés dans un pipeline.
- Random Forest utilise directement les variables non standardisées.
- `class_weight` compense partiellement le déséquilibre des classes.
- Le MLP n'utilise pas `early_stopping=True` : le mécanisme interne de validation aléatoire de scikit-learn ne serait pas conscient des groupes. Le notebook utilise à la place la validation groupée externe pour sélectionner le seuil.


In [3]:
def build_models(seed):
    return {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=seed,
            )),
        ]),

        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced_subsample",
            random_state=seed,
            n_jobs=-1,
        ),

        "MLP": Pipeline([
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                hidden_layer_sizes=(64, 32),
                activation="relu",
                solver="adam",
                max_iter=300,
                early_stopping=False,
                n_iter_no_change=15,
                random_state=seed,
            )),
        ]),
    }


MODEL_NAMES = list(build_models(SEEDS[0]).keys())
print("Modèles :", MODEL_NAMES)


Modèles : ['Logistic Regression', 'Random Forest', 'MLP']


## 5. Sélection du seuil sur la validation

Le seuil retenu maximise le F1-score sur les seules observations de validation. La fonction traite explicitement le cas où aucun seuil n'est disponible.


In [4]:
def select_threshold(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)

    if len(thresholds) == 0:
        return 0.5, float("nan")

    denominator = precision[:-1] + recall[:-1]
    f1_values = np.divide(
        2 * precision[:-1] * recall[:-1],
        denominator,
        out=np.zeros_like(denominator),
        where=denominator != 0,
    )

    best_index = int(np.nanargmax(f1_values))
    return float(thresholds[best_index]), float(f1_values[best_index])


print("Fonction de sélection du seuil définie.")


Fonction de sélection du seuil définie.


## 6. Exécution d'un fold

Pour chaque fold externe :
1. isoler le train externe et le test externe ;
2. créer une validation interne groupée à partir du train externe ;
3. entraîner le modèle uniquement sur le train interne ;
4. sélectionner le seuil sur la validation interne ;
5. évaluer une seule fois sur le test externe.

La matrice de confusion est enregistrée sous la forme `[[TN, FP], [FN, TP]]`.


In [5]:
def run_one_experiment(dataset, model_name, fold_number, seed):
    X = dataset["X"]
    y = dataset["y"]
    groups = dataset["groups"]

    outer_train_idx, outer_test_idx = dataset["outer_splits"][fold_number - 1]

    X_outer_train = X.iloc[outer_train_idx]
    y_outer_train = y[outer_train_idx]
    groups_outer_train = groups[outer_train_idx]

    X_outer_test = X.iloc[outer_test_idx]
    y_outer_test = y[outer_test_idx]

    # Validation interne groupée, limitée au train du fold externe.
    inner_cv = StratifiedGroupKFold(
        n_splits=N_INNER_FOLDS,
        shuffle=True,
        random_state=seed,
    )
    inner_train_idx, inner_val_idx = next(
        inner_cv.split(X_outer_train, y_outer_train, groups_outer_train)
    )

    X_train = X_outer_train.iloc[inner_train_idx]
    y_train = y_outer_train[inner_train_idx]
    X_val = X_outer_train.iloc[inner_val_idx]
    y_val = y_outer_train[inner_val_idx]

    train_groups = np.unique(groups_outer_train[inner_train_idx])
    val_groups = np.unique(groups_outer_train[inner_val_idx])
    test_groups = np.unique(groups[outer_test_idx])

    # Trois ensembles de groupes strictement disjoints.
    assert len(np.intersect1d(train_groups, val_groups)) == 0
    assert len(np.intersect1d(train_groups, test_groups)) == 0
    assert len(np.intersect1d(val_groups, test_groups)) == 0

    model = build_models(seed)[model_name]

    start_time = time.perf_counter()
    model.fit(X_train, y_train)
    training_time = time.perf_counter() - start_time

    # Le seuil est choisi sur la validation uniquement.
    val_probabilities = model.predict_proba(X_val)[:, 1]
    threshold, validation_f1 = select_threshold(y_val, val_probabilities)

    # Le test externe est utilisé uniquement pour l'évaluation finale.
    test_probabilities = model.predict_proba(X_outer_test)[:, 1]
    test_predictions = (test_probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_outer_test, test_predictions, labels=[0, 1]
    ).ravel()

    return {
        "dataset": dataset["name"],
        "model": model_name,
        "fold": fold_number,
        "seed": seed,
        "n_train": len(X_train),
        "n_validation": len(X_val),
        "n_test": len(X_outer_test),
        "threshold": threshold,
        "validation_f1": validation_f1,
        "test_f1": f1_score(y_outer_test, test_predictions, zero_division=0),
        "test_pr_auc": average_precision_score(y_outer_test, test_probabilities),
        "test_precision": precision_score(y_outer_test, test_predictions, zero_division=0),
        "test_recall": recall_score(y_outer_test, test_predictions, zero_division=0),
        "test_accuracy": accuracy_score(y_outer_test, test_predictions),
        "test_balanced_accuracy": balanced_accuracy_score(y_outer_test, test_predictions),
        "training_time_s": training_time,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


print("Fonction d'entraînement et d'évaluation définie.")


Fonction d'entraînement et d'évaluation définie.


## 7. Essai de contrôle (CIC, Logistic Regression, fold 1, seed 42)

Cette cellule reproduit l'essai pilote déjà réalisé. Elle est utile pour vérifier que le notebook complet fonctionne avant de lancer tous les modèles.


In [6]:
pilot_result = run_one_experiment(
    dataset=cic_data,
    model_name="Logistic Regression",
    fold_number=1,
    seed=42,
)

display(pd.DataFrame([pilot_result]).T.rename(columns={0: "Valeur"}))

print("Matrice de confusion [[TN, FP], [FN, TP]] :")
print(np.array([
    [pilot_result["tn"], pilot_result["fp"]],
    [pilot_result["fn"], pilot_result["tp"]],
]))


,Valeur
dataset,CIC-IDS2017
model,Logistic Regression
fold,1
seed,42
n_train,64000
n_validation,15999
n_test,20001
threshold,0.473594
validation_f1,0.631799
test_f1,0.530716


Matrice de confusion [[TN, FP], [FN, TP]] :
[[12778  3265]
 [ 1349  2609]]


## 8. Exécution complète avec sauvegarde progressive

Configuration complète : 2 datasets × 3 modèles × 5 folds × 3 seeds, soit 90 entraînements.

Le fichier CSV est mis à jour après chaque expérience. En cas d'interruption, la cellule peut être relancée : les combinaisons déjà présentes (dataset, modèle, fold, seed) sont ignorées. Pour recommencer entièrement, supprimer ou renommer le fichier de résultats avant de relancer.


In [7]:
RESULTS_FILE = TABLES_DIR / "intra_dataset_baselines_v1.csv"

if RESULTS_FILE.exists():
    results_df = pd.read_csv(RESULTS_FILE)
    completed = set(
        zip(results_df["dataset"], results_df["model"],
            results_df["fold"], results_df["seed"])
    )
    print(f"Reprise : {len(completed)} expériences déjà enregistrées.")
else:
    results_df = pd.DataFrame()
    completed = set()

datasets_to_run = [cic_data, unsw_data]

if RUN_FULL_EXPERIMENT:
    total = len(datasets_to_run) * len(MODEL_NAMES) * N_OUTER_FOLDS * len(SEEDS)
    current = len(completed)
    print(f"Expériences déjà terminées : {current}/{total}")

    for dataset in datasets_to_run:
        for model_name in MODEL_NAMES:
            for fold_number in range(1, N_OUTER_FOLDS + 1):
                for seed in SEEDS:
                    key = (dataset["name"], model_name, fold_number, seed)
                    if key in completed:
                        continue

                    print(
                        f"Exécution : {dataset['name']} | {model_name} | "
                        f"fold={fold_number} | seed={seed}"
                    )
                    result = run_one_experiment(
                        dataset, model_name, fold_number, seed
                    )

                    results_df = pd.concat(
                        [results_df, pd.DataFrame([result])],
                        ignore_index=True,
                    )
                    results_df.to_csv(RESULTS_FILE, index=False)
                    completed.add(key)

    print("\nExpérience complète terminée.")
    print("Résultats sauvegardés dans :", RESULTS_FILE)
else:
    print("RUN_FULL_EXPERIMENT=False : seule l'expérience pilote a été exécutée.")


Expériences déjà terminées : 0/90
Exécution : CIC-IDS2017 | Logistic Regression | fold=1 | seed=42
Exécution : CIC-IDS2017 | Logistic Regression | fold=1 | seed=123
Exécution : CIC-IDS2017 | Logistic Regression | fold=1 | seed=2026
Exécution : CIC-IDS2017 | Logistic Regression | fold=2 | seed=42
Exécution : CIC-IDS2017 | Logistic Regression | fold=2 | seed=123
Exécution : CIC-IDS2017 | Logistic Regression | fold=2 | seed=2026
Exécution : CIC-IDS2017 | Logistic Regression | fold=3 | seed=42
Exécution : CIC-IDS2017 | Logistic Regression | fold=3 | seed=123
Exécution : CIC-IDS2017 | Logistic Regression | fold=3 | seed=2026
Exécution : CIC-IDS2017 | Logistic Regression | fold=4 | seed=42
Exécution : CIC-IDS2017 | Logistic Regression | fold=4 | seed=123
Exécution : CIC-IDS2017 | Logistic Regression | fold=4 | seed=2026
Exécution : CIC-IDS2017 | Logistic Regression | fold=5 | seed=42
Exécution : CIC-IDS2017 | Logistic Regression | fold=5 | seed=123
Exécution : CIC-IDS2017 | Logistic Regressi

D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Exécution : CIC-IDS2017 | MLP | fold=4 | seed=2026
Exécution : CIC-IDS2017 | MLP | fold=5 | seed=42


D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Exécution : CIC-IDS2017 | MLP | fold=5 | seed=123
Exécution : CIC-IDS2017 | MLP | fold=5 | seed=2026


D:\Master-SDIA\CYB-1-IDS-Generalization\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Exécution : UNSW-NB15 | Logistic Regression | fold=1 | seed=42
Exécution : UNSW-NB15 | Logistic Regression | fold=1 | seed=123
Exécution : UNSW-NB15 | Logistic Regression | fold=1 | seed=2026
Exécution : UNSW-NB15 | Logistic Regression | fold=2 | seed=42
Exécution : UNSW-NB15 | Logistic Regression | fold=2 | seed=123
Exécution : UNSW-NB15 | Logistic Regression | fold=2 | seed=2026
Exécution : UNSW-NB15 | Logistic Regression | fold=3 | seed=42
Exécution : UNSW-NB15 | Logistic Regression | fold=3 | seed=123
Exécution : UNSW-NB15 | Logistic Regression | fold=3 | seed=2026
Exécution : UNSW-NB15 | Logistic Regression | fold=4 | seed=42
Exécution : UNSW-NB15 | Logistic Regression | fold=4 | seed=123
Exécution : UNSW-NB15 | Logistic Regression | fold=4 | seed=2026
Exécution : UNSW-NB15 | Logistic Regression | fold=5 | seed=42
Exécution : UNSW-NB15 | Logistic Regression | fold=5 | seed=123
Exécution : UNSW-NB15 | Logistic Regression | fold=5 | seed=2026
Exécution : UNSW-NB15 | Random Forest | 

## 9. Résultats détaillés

Chaque ligne correspond à un dataset, un modèle, un fold externe et une graine.


In [8]:
if RESULTS_FILE.exists():
    results_df = pd.read_csv(RESULTS_FILE)
    display(
        results_df.sort_values(["dataset", "model", "fold", "seed"])
        .reset_index(drop=True)
    )
else:
    print("Aucun fichier de résultats complet pour le moment.")


,dataset,model,fold,seed,n_train,n_validation,n_test,threshold,validation_f1,test_f1,test_pr_auc,test_precision,test_recall,test_accuracy,test_balanced_accuracy,training_time_s,tn,fp,fn,tp
0,CIC-IDS2017,Logistic Regression,1,42,64000,15999,20001,0.473594,0.631799,0.530716,0.483682,0.444161,0.659171,0.769312,0.727828,0.282544,12778,3265,1349,2609
1,CIC-IDS2017,Logistic Regression,1,123,64000,15999,20001,0.476348,0.622754,0.533837,0.487779,0.447851,0.660687,0.771661,0.729864,0.343620,12819,3224,1343,2615
2,CIC-IDS2017,Logistic Regression,1,2026,64000,15999,20001,0.472706,0.649515,0.598608,0.491706,0.469414,0.825922,0.780811,0.797802,0.219108,12348,3695,689,3269
3,CIC-IDS2017,Logistic Regression,2,42,64002,15999,19999,0.468737,0.662765,0.648881,0.528678,0.515314,0.875916,0.812441,0.836350,0.202011,12782,3260,491,3466
4,CIC-IDS2017,Logistic Regression,2,123,64002,15999,19999,0.469948,0.665753,0.617796,0.532587,0.521769,0.757139,0.814641,0.792982,0.169798,13296,2746,961,2996
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,UNSW-NB15,Random Forest,4,123,63539,16452,20009,1.000000,1.000000,0.061538,0.967949,1.000000,0.031746,0.878055,0.515873,5.393744,17489,0,2440,80
86,UNSW-NB15,Random Forest,4,2026,63539,16452,20009,0.985000,0.999595,0.147005,0.971632,0.995025,0.079365,0.884002,0.539654,5.289034,17488,1,2320,200
87,UNSW-NB15,Random Forest,5,42,63553,16452,19995,0.972073,0.998584,0.763681,0.984645,0.996159,0.619180,0.951838,0.809419,5.564404,17476,6,957,1556
88,UNSW-NB15,Random Forest,5,123,63553,16452,19995,0.977201,0.998988,0.756410,0.986179,0.994167,0.610426,0.950588,0.804955,5.841668,17473,9,979,1534


## 10. Agrégation : moyenne et écart-type

Les métriques sont agrégées sur les folds et les graines. La moyenne ± écart-type décrit la variabilité observée dans ce protocole. Les résultats issus de folds qui se recouvrent en observations ne sont pas des répétitions totalement indépendantes ; l'écart-type doit donc être interprété comme une mesure descriptive de stabilité, et non comme un intervalle de confiance statistique.


In [9]:
if RESULTS_FILE.exists():
    results_df = pd.read_csv(RESULTS_FILE)

    metric_columns = [
        "test_f1",
        "test_pr_auc",
        "test_precision",
        "test_recall",
        "test_accuracy",
        "test_balanced_accuracy",
        "training_time_s",
    ]

    summary = (
        results_df
        .groupby(["dataset", "model"])[metric_columns]
        .agg(["mean", "std"])
        .round(4)
    )

    display(summary)

    summary_path = TABLES_DIR / "intra_dataset_baselines_summary_v1.csv"
    summary.to_csv(summary_path)
    print("Synthèse sauvegardée :", summary_path)
else:
    print("Exécuter d'abord l'expérience complète.")


test_f1         test_pr_auc          \
                                   mean     std        mean     std   
dataset     model                                                     
CIC-IDS2017 Logistic Regression  0.6062  0.0350      0.5157  0.0402   
            MLP                  0.7693  0.0337      0.8638  0.0302   
            Random Forest        0.9536  0.0237      0.9825  0.0097   
UNSW-NB15   Logistic Regression  0.4261  0.2216      0.4074  0.0910   
            MLP                  0.6066  0.3365      0.9439  0.0298   
            Random Forest        0.5732  0.3822      0.9815  0.0094   

                                test_precision         test_recall          \
                                          mean     std        mean     std   
dataset     model                                                            
CIC-IDS2017 Logistic Regression         0.4937  0.0249      0.7871  0.0674   
            MLP                         0.6972  0.0977      0.8841  0.1016   
            Random Forest               0.9402  0.0453      0.9687  0.0046   
UNSW-NB15   Logistic Regression         0.3705  0.1555      0.5170  0.3308   
            MLP                         0.8831  0.1355      0.5558  0.3896   
            Random Forest               0.9676  0.0277      0.5246  0.4229   

                                test_accuracy         test_balanced_accuracy  \
                                         mean     std                   mean   
dataset     model                                                              
CIC-IDS2017 Logistic Regression        0.7982  0.0155                 0.7940   
            MLP                        0.8944  0.0229                 0.8905   
            Random Forest              0.9811  0.0102                 0.9764   
UNSW-NB15   Logistic Regression        0.8451  0.0416                 0.7047   
            MLP                        0.9366  0.0439                 0.7736   
            Random Forest              0.9367  0.0495                 0.7603   

                                        training_time_s           
                                    std            mean      std  
dataset     model                                                 
CIC-IDS2017 Logistic Regression  0.0320          0.1857   0.0626  
            MLP                  0.0339        161.6831  55.5062  
            Random Forest        0.0066          5.3659   0.2073  
UNSW-NB15   Logistic Regression  0.1654          0.1872   0.0739  
            MLP                  0.1918         86.1294  35.1100  
            Random Forest        0.2093          5.5205   0.1525

Synthèse sauvegardée : D:\Master-SDIA\CYB-1-IDS-Generalization\results\tables\intra_dataset_baselines_summary_v1.csv


## 11. Vérifications de complétude

L'expérience attendue contient 90 combinaisons uniques. Si le notebook a été interrompu, cette cellule indique les combinaisons manquantes.


In [10]:
if RESULTS_FILE.exists():
    results_df = pd.read_csv(RESULTS_FILE)

    expected = {
        (dataset_name, model_name, fold, seed)
        for dataset_name in ["CIC-IDS2017", "UNSW-NB15"]
        for model_name in MODEL_NAMES
        for fold in range(1, N_OUTER_FOLDS + 1)
        for seed in SEEDS
    }
    observed = set(
        zip(results_df["dataset"], results_df["model"],
            results_df["fold"], results_df["seed"])
    )

    missing = sorted(expected - observed)
    duplicates = results_df.duplicated(
        subset=["dataset", "model", "fold", "seed"]
    ).sum()

    print("Combinaisons attendues :", len(expected))
    print("Combinaisons obtenues :", len(observed))
    print("Combinaisons manquantes :", len(missing))
    print("Lignes dupliquées :", int(duplicates))

    if missing:
        print("Premières combinaisons manquantes :", missing[:10])
    else:
        print("Toutes les expériences attendues sont présentes.")
else:
    print("Fichier de résultats absent.")


Combinaisons attendues : 90
Combinaisons obtenues : 90
Combinaisons manquantes : 0
Lignes dupliquées : 0
Toutes les expériences attendues sont présentes.


## 12. Interprétation et limites

À la fin de l'exécution, interpréter les résultats avec les précautions suivantes :

1. **F1 et PR-AUC sont complémentaires.** Le F1 dépend du seuil choisi ; l'Average Precision (nommée ici `test_pr_auc`) résume la courbe précision-rappel sur différents seuils.
2. **Le seuil n'est pas fixé à 0,5 par défaut.** Il est choisi sur la validation interne, puis appliqué au test externe.
3. **La précision et le rappel décrivent des compromis différents.** Un rappel élevé peut s'accompagner de nombreux faux positifs.
4. **Les résultats intra-dataset ne répondent pas encore à la question de généralisation cross-dataset.** Ils servent de références internes avant CIC → UNSW et UNSW → CIC.
5. **Le schéma de cinq variables est volontairement restreint.** Les performances ne représentent pas celles d'un IDS utilisant toutes les variables disponibles.
6. **Les groupes ne sont pas des sessions réseau.** La séparation protège contre le partage de doublons exacts sur les cinq caractéristiques, mais pas contre toute dépendance possible entre observations.

L'essai pilote précédent (CIC-IDS2017, Logistic Regression, fold 1, seed 42) a donné un F1 test de 0,5307 et une Average Precision de 0,4837. Le résultat complet doit être recalculé par ce notebook pour constituer la table de référence versionnée.


## 13. Fichiers générés

Après l'exécution complète :

- `results/tables/intra_dataset_baselines_v1.csv` : résultats détaillés par fold et seed ;
- `results/tables/intra_dataset_baselines_summary_v1.csv` : moyennes et écarts-types par dataset et modèle.

**Étape scientifique suivante :** analyser les baselines intra-dataset, puis mettre en place les expériences de transfert CIC → UNSW et UNSW → CIC en sélectionnant le seuil uniquement sur la validation du dataset source.


In [11]:

import pandas as pd
from pathlib import Path

results_path = Path("../results/tables/intra_dataset_baselines_v1.csv")
df = pd.read_csv(results_path)

# 1. Vérifier le nombre de résultats
print("Nombre de résultats :", len(df))

# 2. Examiner les performances par fold sur UNSW
unsw = df[df["dataset"] == "UNSW-NB15"]

display(
    unsw.groupby(["model", "fold"])[
        ["threshold", "validation_f1", "test_f1", "test_pr_auc"]
    ].agg(["mean", "std"]).round(4)
)

# 3. Examiner les seuils sélectionnés
display(
    unsw.groupby("model")["threshold"]
    .agg(["min", "max", "mean", "std"])
    .round(4)
)

# 4. Vérifier la relation entre PR-AUC et F1
display(
    unsw.groupby("model")[
        ["test_f1", "test_pr_auc", "test_precision", "test_recall"]
    ].agg(["mean", "std"]).round(4)
)



Nombre de résultats : 90


threshold         validation_f1         test_f1  \
                              mean     std          mean     std    mean   
model               fold                                                   
Logistic Regression 1       0.6728  0.0021        0.5699  0.0016  0.7226   
                    2       0.7073  0.0001        0.7780  0.0040  0.3765   
                    3       0.6739  0.0010        0.5510  0.0010  0.6241   
                    4       0.7118  0.0001        0.7810  0.0011  0.1866   
                    5       0.7098  0.0007        0.7825  0.0021  0.2207   
MLP                 1       0.4661  0.0342        0.9312  0.0068  0.9359   
                    2       0.9541  0.0270        0.9927  0.0075  0.4789   
                    3       0.4145  0.0674        0.9250  0.0081  0.9260   
                    4       0.8929  0.1414        0.9964  0.0028  0.1328   
                    5       0.9629  0.0087        0.9976  0.0015  0.5594   
Random Forest       1       0.3359  0.0102        0.9548  0.0044  0.9643   
                    2       0.9731  0.0211        0.9993  0.0005  0.2473   
                    3       0.3450  0.0346        0.9506  0.0045  0.9520   
                    4       0.9950  0.0086        0.9997  0.0002  0.0954   
                    5       0.9794  0.0086        0.9990  0.0004  0.6067   

                                 test_pr_auc          
                             std        mean     std  
model               fold                              
Logistic Regression 1     0.0003      0.5815  0.0003  
                    2     0.0001      0.3508  0.0004  
                    3     0.0001      0.3787  0.0002  
                    4     0.0010      0.3490  0.0005  
                    5     0.0002      0.3772  0.0005  
MLP                 1     0.0015      0.9602  0.0239  
                    2     0.0708      0.9476  0.0222  
                    3     0.0063      0.9474  0.0026  
                    4     0.0792      0.9152  0.0584  
                    5     0.3177      0.9494  0.0060  
Random Forest       1     0.0013      0.9950  0.0013  
                    2     0.0415      0.9753  0.0016  
                    3     0.0028      0.9824  0.0028  
                    4     0.0454      0.9688  0.0025  
                    5     0.2656      0.9860  0.0013

,min,max,mean,std
model,,,,
Logistic Regression,0.6706,0.7119,0.6951,0.0185
MLP,0.3369,0.9848,0.7381,0.2608
Random Forest,0.3050,1.0000,0.7257,0.3261


test_f1         test_pr_auc         test_precision  \
                       mean     std        mean     std           mean   
model                                                                    
Logistic Regression  0.4261  0.2216      0.4074  0.0910         0.3705   
MLP                  0.6066  0.3365      0.9439  0.0298         0.8831   
Random Forest        0.5732  0.3822      0.9815  0.0094         0.9676   

                            test_recall          
                        std        mean     std  
model                                            
Logistic Regression  0.1555      0.5170  0.3308  
MLP                  0.1355      0.5558  0.3896  
Random Forest        0.0277      0.5246  0.4229

In [12]:

# Analyse détaillée des seuils sur UNSW-NB15

unsw = df[df["dataset"] == "UNSW-NB15"].copy()

print("=== Seuils sélectionnés ===")
display(
    unsw.groupby("model")["threshold"]
    .agg(["min", "max", "mean", "std"])
    .round(4)
)

print("=== Résultats par fold ===")
display(
    unsw.groupby(["model", "fold"])[
        ["threshold", "validation_f1", "test_f1",
         "test_precision", "test_recall", "test_pr_auc"]
    ].mean().round(4)
)

print("=== Résultats individuels ===")
display(
    unsw[
        ["model", "fold", "seed", "threshold",
         "validation_f1", "test_f1",
         "test_precision", "test_recall", "test_pr_auc"]
    ].sort_values(["model", "fold", "seed"])
)


=== Seuils sélectionnés ===


,min,max,mean,std
model,,,,
Logistic Regression,0.6706,0.7119,0.6951,0.0185
MLP,0.3369,0.9848,0.7381,0.2608
Random Forest,0.3050,1.0000,0.7257,0.3261


=== Résultats par fold ===


threshold  validation_f1  test_f1  test_precision  \
model               fold                                                      
Logistic Regression 1        0.6728         0.5699   0.7226          0.5702   
                    2        0.7073         0.7780   0.3765          0.3441   
                    3        0.6739         0.5510   0.6241          0.5158   
                    4        0.7118         0.7810   0.1866          0.1967   
                    5        0.7098         0.7825   0.2207          0.2258   
MLP                 1        0.4661         0.9312   0.9359          0.8824   
                    2        0.9541         0.9927   0.4789          0.9614   
                    3        0.4145         0.9250   0.9260          0.9034   
                    4        0.8929         0.9964   0.1328          0.7369   
                    5        0.9629         0.9976   0.5594          0.9316   
Random Forest       1        0.3359         0.9548   0.9643          0.9338   
                    2        0.9731         0.9993   0.2473          0.9744   
                    3        0.3450         0.9506   0.9520          0.9395   
                    4        0.9950         0.9997   0.0954          0.9951   
                    5        0.9794         0.9990   0.6067          0.9953   

                          test_recall  test_pr_auc  
model               fold                            
Logistic Regression 1          0.9859       0.5815  
                    2          0.4157       0.3508  
                    3          0.7901       0.3787  
                    4          0.1775       0.3490  
                    5          0.2158       0.3772  
MLP                 1          0.9963       0.9602  
                    2          0.3214       0.9476  
                    3          0.9500       0.9474  
                    4          0.0749       0.9152  
                    5          0.4365       0.9494  
Random Forest       1          0.9969       0.9950  
                    2          0.1421       0.9753  
                    3          0.9649       0.9824  
                    4          0.0505       0.9688  
                    5          0.4688       0.9860

=== Résultats individuels ===


,model,fold,seed,threshold,validation_f1,test_f1,test_precision,test_recall,test_pr_auc
45,Logistic Regression,1,42,0.672972,0.569085,0.722660,0.570311,0.986072,0.581658
46,Logistic Regression,1,123,0.674837,0.568913,0.722790,0.570606,0.985674,0.581658
47,Logistic Regression,1,2026,0.670591,0.571835,0.722238,0.569786,0.986072,0.581137
48,Logistic Regression,2,42,0.707375,0.780218,0.376619,0.344068,0.415971,0.350958
49,Logistic Regression,2,123,0.707218,0.780341,0.376462,0.344079,0.415574,0.351147
50,Logistic Regression,2,2026,0.707410,0.773375,0.376462,0.344079,0.415574,0.350379
51,Logistic Regression,3,42,0.673215,0.551673,0.624254,0.515836,0.790374,0.378443
52,Logistic Regression,3,123,0.675076,0.551456,0.624038,0.515710,0.789976,0.378867
53,Logistic Regression,3,2026,0.673527,0.549805,0.624038,0.515710,0.789976,0.378826
54,Logistic Regression,4,42,0.711914,0.780835,0.186483,0.196570,0.177381,0.348744


In [13]:

# Diagnostic des distributions de classes - UNSW-NB15

diagnostic_rows = []

X = unsw_data["X"]
y = unsw_data["y"]
groups = unsw_data["groups"]

for fold_number, (outer_train_idx, outer_test_idx) in enumerate(
    unsw_data["outer_splits"], start=1
):
    y_outer_train = y[outer_train_idx]
    groups_outer_train = groups[outer_train_idx]

    y_outer_test = y[outer_test_idx]

    for seed in SEEDS:
        inner_cv = StratifiedGroupKFold(
            n_splits=N_INNER_FOLDS,
            shuffle=True,
            random_state=seed,
        )

        inner_train_idx, inner_val_idx = next(
            inner_cv.split(
                X.iloc[outer_train_idx],
                y_outer_train,
                groups_outer_train
            )
        )

        y_train = y_outer_train[inner_train_idx]
        y_val = y_outer_train[inner_val_idx]

        diagnostic_rows.append({
            "fold": fold_number,
            "seed": seed,

            "outer_train_size": len(outer_train_idx),
            "outer_train_attack_rate": y_outer_train.mean(),

            "inner_train_size": len(inner_train_idx),
            "inner_train_attack_rate": y_train.mean(),

            "validation_size": len(inner_val_idx),
            "validation_attack_rate": y_val.mean(),

            "test_size": len(outer_test_idx),
            "test_attack_rate": y_outer_test.mean(),

            "train_groups": len(
                np.unique(groups_outer_train[inner_train_idx])
            ),
            "validation_groups": len(
                np.unique(groups_outer_train[inner_val_idx])
            ),
            "test_groups": len(
                np.unique(groups[outer_test_idx])
            ),
        })

diagnostic_df = pd.DataFrame(diagnostic_rows)

display(
    diagnostic_df.round(4).sort_values(["fold", "seed"])
)

print("\nMoyenne des proportions d'attaques par fold :")

display(
    diagnostic_df.groupby("fold")[
        [
            "outer_train_attack_rate",
            "inner_train_attack_rate",
            "validation_attack_rate",
            "test_attack_rate",
        ]
    ].mean().round(4)
)


,fold,seed,outer_train_size,outer_train_attack_rate,inner_train_size,inner_train_attack_rate,validation_size,validation_attack_rate,test_size,test_attack_rate,train_groups,validation_groups,test_groups
0,1,42,80004,0.1258,64005,0.1258,15999,0.1258,19996,0.1257,41410,10365,12342
1,1,123,80004,0.1258,64001,0.1258,16003,0.1259,19996,0.1257,41405,10370,12342
2,1,2026,80004,0.1258,63995,0.1257,16009,0.1260,19996,0.1257,41413,10362,12342
3,2,42,79997,0.1258,63545,0.1195,16452,0.1501,20003,0.1258,41331,9838,12948
4,2,123,79997,0.1258,63545,0.1195,16452,0.1501,20003,0.1258,41330,9839,12948
5,2,2026,79997,0.1258,63545,0.1195,16452,0.1501,20003,0.1258,41331,9838,12948
6,3,42,80003,0.1258,64112,0.1273,15891,0.1196,19997,0.1257,40886,10296,12935
7,3,123,80003,0.1258,64114,0.1273,15889,0.1195,19997,0.1257,40850,10332,12935
8,3,2026,80003,0.1258,64118,0.1274,15885,0.1194,19997,0.1257,40834,10348,12935
9,4,42,79991,0.1257,63539,0.1194,16452,0.1501,20009,0.1259,41352,9838,12927



Moyenne des proportions d'attaques par fold :


,outer_train_attack_rate,inner_train_attack_rate,validation_attack_rate,test_attack_rate
fold,,,,
1,0.1258,0.1258,0.1259,0.1257
2,0.1258,0.1195,0.1501,0.1258
3,0.1258,0.1273,0.1195,0.1257
4,0.1257,0.1194,0.1501,0.1259
5,0.1258,0.1195,0.1501,0.1257


In [14]:

# Diagnostic des groupes UNSW-NB15
# Vérifier la composition des groupes et leur influence
# sur la stratification interne.

import numpy as np
import pandas as pd

# Distribution des classes par groupe
group_df = pd.DataFrame({
    "group": groups,
    "target": y
})

group_stats = (
    group_df.groupby("group")
    .agg(
        group_size=("target", "size"),
        attack_count=("target", "sum"),
        attack_rate=("target", "mean")
    )
)

print("=== Statistiques des groupes ===")
display(group_stats.describe().round(4))

print("\n=== Répartition des groupes selon leur composition ===")

group_stats["group_type"] = np.select(
    [
        group_stats["attack_count"] == 0,
        group_stats["attack_count"] == group_stats["group_size"]
    ],
    [
        "Normal uniquement",
        "Attaques uniquement"
    ],
    default="Groupes mixtes"
)

display(
    group_stats.groupby("group_type")
    .agg(
        number_of_groups=("group_size", "size"),
        total_observations=("group_size", "sum"),
        mean_group_size=("group_size", "mean"),
        max_group_size=("group_size", "max")
    )
)

print("\n=== Les 20 groupes les plus grands ===")
display(
    group_stats.sort_values("group_size", ascending=False)
    .head(20)
)


=== Statistiques des groupes ===


,group_size,attack_count,attack_rate
count,64117.0000,64117.0000,64117.0000
mean,1.5596,0.1962,0.0421
std,17.6820,12.8768,0.2005
min,1.0000,0.0000,0.0000
25%,1.0000,0.0000,0.0000
50%,1.0000,0.0000,0.0000
75%,1.0000,0.0000,0.0000
max,2469.0000,2469.0000,1.0000



=== Répartition des groupes selon leur composition ===


,number_of_groups,total_observations,mean_group_size,max_group_size
group_type,,,,
Attaques uniquement,2676,12197,4.557922,2469
Groupes mixtes,50,591,11.820000,106
Normal uniquement,61391,87212,1.420599,1814



=== Les 20 groupes les plus grands ===


,group_size,attack_count,attack_rate,group_type
group,,,,
24,2469,2469,1.0,Attaques uniquement
137,1814,0,0.0,Normal uniquement
55,1198,1198,1.0,Attaques uniquement
100,1125,0,0.0,Normal uniquement
70,1069,0,0.0,Normal uniquement
123,1060,0,0.0,Normal uniquement
174,1054,1054,1.0,Attaques uniquement
120,912,0,0.0,Normal uniquement
86,663,0,0.0,Normal uniquement


In [19]:
# Afficher les variables du notebook contenant "split"
for name in globals():
    if "split" in name.lower() or "fold" in name.lower():
        obj = globals()[name]
        print(name, type(obj))

StratifiedGroupKFold <class 'abc.ABCMeta'>


RuntimeError: dictionary changed size during iteration

In [20]:
for name, obj in globals().copy().items():
    if isinstance(obj, np.ndarray):
        if obj.ndim >= 1:
            print(name, obj.shape, obj.dtype)

y (100000,) int64
groups (100000,) int64
outer_train_idx (80005,) int64
outer_test_idx (19995,) int64
y_outer_train (80005,) int64
groups_outer_train (80005,) int64
y_outer_test (19995,) int64
inner_train_idx (63553,) int64
inner_val_idx (16452,) int64
y_train (63553,) int64
y_val (16452,) int64


In [21]:
# Rechercher les variables liées à UNSW et aux splits
for name, obj in list(globals().copy().items()):
    if any(word in name.lower() for word in ["unsw", "outer", "split", "fold"]):
        print(
            name,
            type(obj),
            getattr(obj, "shape", ""),
        )

StratifiedGroupKFold <class 'abc.ABCMeta'> 
N_OUTER_FOLDS <class 'int'> 
N_INNER_FOLDS <class 'int'> 
unsw_data <class 'dict'> 
fold_number <class 'int'> 
unsw <class 'pandas.DataFrame'> (45, 20)
outer_train_idx <class 'numpy.ndarray'> (80005,)
outer_test_idx <class 'numpy.ndarray'> (19995,)
y_outer_train <class 'numpy.ndarray'> (80005,)
groups_outer_train <class 'numpy.ndarray'> (80005,)
y_outer_test <class 'numpy.ndarray'> (19995,)


In [25]:
print("Type :", type(unsw_data))
print("Clés disponibles :", list(unsw_data.keys()))

for key, value in unsw_data.items():
    print(
        f"{key}: type={type(value)}, "
        f"shape={getattr(value, 'shape', None)}"
    )

Type : <class 'dict'>
Clés disponibles : ['name', 'X', 'y', 'groups', 'outer_splits']
name: type=<class 'str'>, shape=None
X: type=<class 'pandas.DataFrame'>, shape=(100000, 5)
y: type=<class 'numpy.ndarray'>, shape=(100000,)
groups: type=<class 'numpy.ndarray'>, shape=(100000,)
outer_splits: type=<class 'list'>, shape=None


In [26]:
print("Fold actuellement chargé :", fold_number)

print("Taille train :", len(outer_train_idx))
print("Taille test :", len(outer_test_idx))

print("Proportion attaques train :",
      round(y_outer_train.mean() * 100, 3), "%")

print("Proportion attaques test :",
      round(y_outer_test.mean() * 100, 3), "%")

Fold actuellement chargé : 5
Taille train : 80005
Taille test : 19995
Proportion attaques train : 12.579 %
Proportion attaques test : 12.568 %


In [27]:
import numpy as np
import pandas as pd

# Récupérer les données originales
X_unsw = unsw_data["X"]
y_unsw = np.asarray(unsw_data["y"])
groups_unsw = np.asarray(unsw_data["groups"])
outer_splits_unsw = unsw_data["outer_splits"]

# Vérifications
assert len(X_unsw) == len(y_unsw) == len(groups_unsw) == 100_000
assert len(outer_splits_unsw) == 5

# Identifier les 20 groupes les plus volumineux
group_stats_unsw = pd.DataFrame({
    "group": groups_unsw,
    "target": y_unsw
}).groupby("group").agg(
    group_size=("target", "size"),
    attack_count=("target", "sum")
)

top_20_groups = group_stats_unsw.nlargest(
    20, "group_size"
).index.to_numpy()

print("Taille totale des 20 plus grands groupes :",
      group_stats_unsw.loc[top_20_groups, "group_size"].sum())

# Diagnostic sur les 5 folds
fold_diagnostics = []

for fold_id, (train_idx, test_idx) in enumerate(
    outer_splits_unsw, start=1
):
    for subset_name, indices in [
        ("Train", train_idx),
        ("Test", test_idx)
    ]:
        subset_groups = groups_unsw[indices]

        top_mask = np.isin(subset_groups, top_20_groups)

        fold_diagnostics.append({
            "fold": fold_id,
            "subset": subset_name,
            "total_observations": len(indices),
            "top20_observations": int(top_mask.sum()),
            "top20_percentage": 100 * top_mask.mean(),
            "attack_rate": 100 * y_unsw[indices].mean(),
            "unique_groups": len(np.unique(subset_groups))
        })

fold_diagnostic_df = pd.DataFrame(fold_diagnostics)

display(fold_diagnostic_df.round(3))

Taille totale des 20 plus grands groupes : 16435


,fold,subset,total_observations,top20_observations,top20_percentage,attack_rate,unique_groups
0,1,Train,80004,12391,15.488,12.579,51775
1,1,Test,19996,4044,20.224,12.568,12342
2,2,Train,79997,13124,16.406,12.575,51169
3,2,Test,20003,3311,16.553,12.583,12948
4,3,Train,80003,13239,16.548,12.578,51182
5,3,Test,19997,3196,15.982,12.572,12935
6,4,Train,79991,13450,16.814,12.573,51190
7,4,Test,20009,2985,14.918,12.594,12927
8,5,Train,80005,13536,16.919,12.579,51152
9,5,Test,19995,2899,14.499,12.568,12965


In [28]:
# Présence de chacun des 20 grands groupes dans chaque fold

large_group_distribution = []

for group_id in top_20_groups:
    original_size = int(
        group_stats_unsw.loc[group_id, "group_size"]
    )

    for fold_id, (train_idx, test_idx) in enumerate(
        outer_splits_unsw, start=1
    ):
        train_count = np.sum(groups_unsw[train_idx] == group_id)
        test_count = np.sum(groups_unsw[test_idx] == group_id)

        large_group_distribution.append({
            "group": int(group_id),
            "original_size": original_size,
            "fold": fold_id,
            "train_count": int(train_count),
            "test_count": int(test_count)
        })

large_group_df = pd.DataFrame(large_group_distribution)

display(
    large_group_df.pivot(
        index=["group", "original_size"],
        columns="fold",
        values="test_count"
    ).fillna(0).astype(int)
)

,fold,1,2,3,4,5
group,original_size,,,,,
24,2469,2469,0,0,0,0
32,596,0,0,596,0,0
55,1198,0,0,1198,0,0
70,1069,0,0,0,0,1069
81,253,0,253,0,0,0
84,659,0,0,0,659,0
86,663,663,0,0,0,0
100,1125,0,0,0,1125,0
111,374,0,0,0,374,0


In [30]:
from pathlib import Path
import pandas as pd

results_path = Path(
    r"D:\Master-SDIA\CYB-1-IDS-Generalization\results\tables\intra_dataset_baselines_v1.csv"
)

assert results_path.exists(), f"Fichier introuvable : {results_path}"

results_df = pd.read_csv(results_path)

print("Fichier chargé :", results_path)
print("Dimensions :", results_df.shape)
print("\nColonnes :")
print(results_df.columns.tolist())

display(results_df.head())

Fichier chargé : D:\Master-SDIA\CYB-1-IDS-Generalization\results\tables\intra_dataset_baselines_v1.csv
Dimensions : (90, 20)

Colonnes :
['dataset', 'model', 'fold', 'seed', 'n_train', 'n_validation', 'n_test', 'threshold', 'validation_f1', 'test_f1', 'test_pr_auc', 'test_precision', 'test_recall', 'test_accuracy', 'test_balanced_accuracy', 'training_time_s', 'tn', 'fp', 'fn', 'tp']


,dataset,model,fold,seed,n_train,n_validation,n_test,threshold,validation_f1,test_f1,test_pr_auc,test_precision,test_recall,test_accuracy,test_balanced_accuracy,training_time_s,tn,fp,fn,tp
0,CIC-IDS2017,Logistic Regression,1,42,64000,15999,20001,0.473594,0.631799,0.530716,0.483682,0.444161,0.659171,0.769312,0.727828,0.282544,12778,3265,1349,2609
1,CIC-IDS2017,Logistic Regression,1,123,64000,15999,20001,0.476348,0.622754,0.533837,0.487779,0.447851,0.660687,0.771661,0.729864,0.343620,12819,3224,1343,2615
2,CIC-IDS2017,Logistic Regression,1,2026,64000,15999,20001,0.472706,0.649515,0.598608,0.491706,0.469414,0.825922,0.780811,0.797802,0.219108,12348,3695,689,3269
3,CIC-IDS2017,Logistic Regression,2,42,64002,15999,19999,0.468737,0.662765,0.648881,0.528678,0.515314,0.875916,0.812441,0.836350,0.202011,12782,3260,491,3466
4,CIC-IDS2017,Logistic Regression,2,123,64002,15999,19999,0.469948,0.665753,0.617796,0.532587,0.521769,0.757139,0.814641,0.792982,0.169798,13296,2746,961,2996


In [31]:
print("Nombre total de lignes :", len(results_df))

print("\nRépartition par dataset et modèle :")
display(
    results_df.groupby(["dataset", "model"])
    .size()
    .reset_index(name="nombre_experiences")
)

print("\nValeurs manquantes :")
display(results_df.isna().sum())

Nombre total de lignes : 90

Répartition par dataset et modèle :


,dataset,model,nombre_experiences
0,CIC-IDS2017,Logistic Regression,15
1,CIC-IDS2017,MLP,15
2,CIC-IDS2017,Random Forest,15
3,UNSW-NB15,Logistic Regression,15
4,UNSW-NB15,MLP,15
5,UNSW-NB15,Random Forest,15



Valeurs manquantes :


dataset                   0
model                     0
fold                      0
seed                      0
n_train                   0
n_validation              0
n_test                    0
threshold                 0
validation_f1             0
test_f1                   0
test_pr_auc               0
test_precision            0
test_recall               0
test_accuracy             0
test_balanced_accuracy    0
training_time_s           0
tn                        0
fp                        0
fn                        0
tp                        0
dtype: int64

In [32]:
# 1. Résultats UNSW uniquement
unsw_results = results_df[
    results_df["dataset"] == "UNSW-NB15"
].copy()

# 2. Moyenne et écart-type des métriques par modèle et fold
fold_performance = (
    unsw_results
    .groupby(["model", "fold"])
    .agg(
        f1_mean=("test_f1", "mean"),
        f1_std=("test_f1", "std"),
        pr_auc_mean=("test_pr_auc", "mean"),
        pr_auc_std=("test_pr_auc", "std"),
        precision_mean=("test_precision", "mean"),
        recall_mean=("test_recall", "mean"),
        threshold_mean=("threshold", "mean")
    )
    .reset_index()
)

# 3. Proportion des 20 grands groupes dans chaque test
top20_test = (
    fold_diagnostic_df[
        fold_diagnostic_df["subset"] == "Test"
    ][["fold", "top20_percentage", "attack_rate"]]
)

# 4. Fusionner les deux tableaux
fold_analysis = fold_performance.merge(
    top20_test,
    on="fold",
    how="left"
)

# 5. Affichage
display(
    fold_analysis.sort_values(
        ["model", "fold"]
    ).round(4)
)

,model,fold,f1_mean,f1_std,pr_auc_mean,pr_auc_std,precision_mean,recall_mean,threshold_mean,top20_percentage,attack_rate
0,Logistic Regression,1,0.7226,0.0003,0.5815,0.0003,0.5702,0.9859,0.6728,20.2240,12.5675
1,Logistic Regression,2,0.3765,0.0001,0.3508,0.0004,0.3441,0.4157,0.7073,16.5525,12.5831
2,Logistic Regression,3,0.6241,0.0001,0.3787,0.0002,0.5158,0.7901,0.6739,15.9824,12.5719
3,Logistic Regression,4,0.1866,0.0010,0.3490,0.0005,0.1967,0.1775,0.7118,14.9183,12.5943
4,Logistic Regression,5,0.2207,0.0002,0.3772,0.0005,0.2258,0.2158,0.7098,14.4986,12.5681
5,MLP,1,0.9359,0.0015,0.9602,0.0239,0.8824,0.9963,0.4661,20.2240,12.5675
6,MLP,2,0.4789,0.0708,0.9476,0.0222,0.9614,0.3214,0.9541,16.5525,12.5831
7,MLP,3,0.9260,0.0063,0.9474,0.0026,0.9034,0.9500,0.4145,15.9824,12.5719
8,MLP,4,0.1328,0.0792,0.9152,0.0584,0.7369,0.0749,0.8929,14.9183,12.5943
9,MLP,5,0.5594,0.3177,0.9494,0.0060,0.9316,0.4365,0.9629,14.4986,12.5681


In [33]:
# Corrélations descriptives entre proportion des grands groupes
# et métriques de performance, séparément par modèle.

correlation_results = []

for model in fold_analysis["model"].unique():

    model_data = fold_analysis[
        fold_analysis["model"] == model
    ]

    correlation_results.append({
        "model": model,
        "spearman_top20_f1": model_data[
            "top20_percentage"
        ].corr(
            model_data["f1_mean"],
            method="spearman"
        ),
        "spearman_top20_pr_auc": model_data[
            "top20_percentage"
        ].corr(
            model_data["pr_auc_mean"],
            method="spearman"
        )
    })

correlation_df = pd.DataFrame(correlation_results)

display(correlation_df.round(4))

,model,spearman_top20_f1,spearman_top20_pr_auc
0,Logistic Regression,0.8,0.5
1,MLP,0.5,0.4
2,Random Forest,0.5,0.3


In [34]:
threshold_analysis = (
    results_df[results_df["dataset"] == "UNSW-NB15"]
    .groupby(["model", "fold"])
    .agg(
        threshold_min=("threshold", "min"),
        threshold_mean=("threshold", "mean"),
        threshold_max=("threshold", "max"),
        validation_f1_mean=("validation_f1", "mean"),
        test_f1_mean=("test_f1", "mean"),
        test_pr_auc_mean=("test_pr_auc", "mean")
    )
    .reset_index()
)

display(
    threshold_analysis
    .sort_values(["model", "fold"])
    .round(4)
)

,model,fold,threshold_min,threshold_mean,threshold_max,validation_f1_mean,test_f1_mean,test_pr_auc_mean
0,Logistic Regression,1,0.6706,0.6728,0.6748,0.5699,0.7226,0.5815
1,Logistic Regression,2,0.7072,0.7073,0.7074,0.7780,0.3765,0.3508
2,Logistic Regression,3,0.6732,0.6739,0.6751,0.5510,0.6241,0.3787
3,Logistic Regression,4,0.7117,0.7118,0.7119,0.7810,0.1866,0.3490
4,Logistic Regression,5,0.7090,0.7098,0.7103,0.7825,0.2207,0.3772
5,MLP,1,0.4427,0.4661,0.5054,0.9312,0.9359,0.9602
6,MLP,2,0.9233,0.9541,0.9738,0.9927,0.4789,0.9476
7,MLP,3,0.3369,0.4145,0.4586,0.9250,0.9260,0.9474
8,MLP,4,0.7300,0.8929,0.9848,0.9964,0.1328,0.9152
9,MLP,5,0.9537,0.9629,0.9709,0.9976,0.5594,0.9494


In [35]:
precision, recall, thresholds = precision_recall_curve(
    y_validation,
    validation_probabilities
)

f1_scores = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

NameError: name 'y_validation' is not defined